# Chunking Experiments & Analysis

In RAG pipelines, chunking is one of the most critical hyperparameters directly determining retrieval precision and context retention.

This notebook demonstrates:
1. Loading sample fitness and sports physiology documents.
2. Evaluating different chunking strategies (`RecursiveCharacterTextSplitter` vs fixed-size).
3. Comparing chunk length distributions for sizes: **256**, **512**, and **1024** characters.
4. Examining chunk boundaries, metadata preservation, and coverage across domain pillars.

In [ ]:
import sys
from pathlib import Path

# Add src to python path
root_dir = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(root_dir / "src"))

from rag_fitness_coach.ingestion.chunker import build_splitter, chunk_documents, chunk_stats
from langchain_core.documents import Document
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

## 1. Prepare Sample Training & Nutrition Texts

In [ ]:
sample_texts = [
    Document(
        page_content=(
            "Hypertrophy Training Principles:\n\n"
            "Muscular hypertrophy occurs primarily through mechanical tension, muscle damage, and metabolic stress. "
            "Progressive mechanical tension is widely regarded as the primary driver of muscle growth. To optimize this, "
            "training volume should typically fall between 10 to 20 weekly sets per muscle group for trained individuals. "
            "Each working set should be taken close to muscular failure, typically within 1 to 3 Repetitions in Reserve (RIR).\n\n"
            "Compound vs Isolation Movements:\n"
            "Multi-joint movements (e.g., barbell squats, deadlifts, overhead presses) recruit large muscle masses and allow greater mechanical loading. "
            "Single-joint movements (e.g., lateral raises, leg extensions) allow targeted loading with minimal systemic fatigue, "
            "making them ideal for addressing muscular imbalances and accumulating targeted volume."
        ),
        metadata={"title": "Principles of Hypertrophy", "pillar": "exercise_library", "source_type": "reference"}
    ),
    Document(
        page_content=(
            "Nutrition Guidelines for Athletic Performance:\n\n"
            "Macronutrient distribution plays a vital role in fueling performance and driving adaptation. "
            "Protein intake should range from 1.6 to 2.2 grams per kilogram of body weight per day for resistance-trained athletes, "
            "spaced across 3 to 5 meals containing at least 20-40g of high-quality protein rich in leucine.\n\n"
            "Carbohydrates serve as the primary fuel source for anaerobic glycolysis during high-intensity training. "
            "Intakes of 3 to 7 g/kg/day are recommended depending on session duration and energy expenditure. "
            "Hydration is equally critical: dehydration exceeding 2% body mass loss significantly impairs strength, cognition, and endurance."
        ),
        metadata={"title": "Sports Nutrition & Macros", "pillar": "nutrition", "source_type": "reference"}
    ),
    Document(
        page_content=(
            "Exercise Considerations for Low Back Pain:\n\n"
            "Non-specific low back pain is one of the most common musculoskeletal complaints globally. "
            "Bed rest is generally contraindicated; progressive active movement and graded physical activity promote faster recovery. "
            "Core stabilization exercises, including the McGill Big Three (Modified Curl-Up, Side Bridge, and Bird Dog), "
            "enhance spinal stiffness without imposing excessive compressive loads on the lumbar vertebrae.\n\n"
            "Aerobic conditioning, such as brisk walking, improves tissue perfusion and reduces central pain sensitization."
        ),
        metadata={"title": "Managing Low Back Pain with Exercise", "pillar": "health_issues", "source_type": "reference"}
    )
]

print(f"Created {len(sample_texts)} documents across {[d.metadata['pillar'] for d in sample_texts]}")

## 2. Compare Chunk Size Profiles (256 vs 512 vs 1024 chars)

In [ ]:
configs = [
    {"size": 256, "overlap": 32},
    {"size": 512, "overlap": 64},
    {"size": 1024, "overlap": 128},
]

results = []
all_chunks_by_config = {}

for cfg in configs:
    c_size, c_overlap = cfg["size"], cfg["overlap"]
    chunks = chunk_documents(sample_texts, chunk_size=c_size, chunk_overlap=c_overlap)
    stats = chunk_stats(chunks)
    stats["config_chunk_size"] = c_size
    stats["config_overlap"] = c_overlap
    results.append(stats)
    all_chunks_by_config[f"{c_size}_{c_overlap}"] = chunks

df_results = pd.DataFrame(results)
df_results[["config_chunk_size", "config_overlap", "total_chunks", "min_chars", "mean_chars", "median_chars", "max_chars"]]

## 3. Inspecting Metadata & Chunk Boundaries

In [ ]:
# Let's inspect the chunks from the 512 character configuration
selected_chunks = all_chunks_by_config["512_64"]
for i, chunk in enumerate(selected_chunks):
    print(f"--- Chunk {i+1} | Pillar: {chunk.metadata.get('pillar')} | Length: {len(chunk.page_content)} chars ---")
    print(chunk.page_content.strip())
    print()
